# 02 Interaction Cleaning

## Purpose
Apply the interaction-cleaning decisions informed by the dataset audit and produce modelling-ready interaction datasets for later recommendation experiments.

## Objectives
- load the raw interaction dataset
- standardise essential interaction fields
- parse dates for chronological use
- remove exact duplicate records if present
- preserve review text without modelling it
- define how `rating = 0` will be treated
- create explicit and implicit interaction views
- evaluate low-signal filtering as a modelling option
- save cleaned outputs for downstream stages

## Audit findings carried into this phase

The dataset audit established the following points that directly affect interaction cleaning:

- the interaction dataset is structurally complete for the essential fields `user_id`, `recipe_id`, `date`, and `rating`
- missingness is negligible and mainly limited to the `review` column
- no full-row duplicates were previously found
- no duplicate user–recipe pairs were found in the audit
- all dates were parseable in the audit
- the rating distribution is highly skewed toward positive feedback
- `rating = 0` is present and requires an explicit modelling decision
- the interaction matrix is extremely sparse and follows a long-tail distribution

This notebook therefore focuses on enforcing cleaning rules rather than repeating the full audit.

In [97]:
from __future__ import annotations
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [98]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [99]:
PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.paths import (
    RAW_INTERACTIONS_PATH,
    INTERIM_DIR,
    PROCESSED_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    ensure_directories,
)

ensure_directories()

RAW_INTERACTIONS_PATH

WindowsPath('E:/UWE/Class Notes/Year 3/Digital Systems Project/Project V2/project/data/raw/RAW_interactions.csv')

## 1. Load raw interactions
Load the raw interaction dataset that will be cleaned in this phase.

In [100]:
df_raw = pd.read_csv(RAW_INTERACTIONS_PATH)
df_raw.head()

,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin..."


In [101]:
print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())

Raw shape: (1132367, 5)
Columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']


## 2. Keep the required interaction columns
Only the fields required for interaction cleaning and later recommendation modelling are retained.

In [102]:
required_columns = ["user_id", "recipe_id", "date", "rating", "review"]
df = df_raw[required_columns].copy()
df.head()

,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin..."


## 3. Standardise essential column types
The key interaction fields are converted into consistent modelling-friendly types.
Rows with invalid values in essential columns are removed.

In [103]:
for col in ["user_id", "recipe_id", "rating"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

invalid_core_rows = df[
    df["user_id"].isna() |
    df["recipe_id"].isna() |
    df["rating"].isna()
].copy()

print("Rows with invalid essential numeric values:", len(invalid_core_rows))
invalid_core_rows.head()

Rows with invalid essential numeric values: 0


,user_id,recipe_id,date,rating,review


In [104]:
df = df.dropna(subset=["user_id", "recipe_id", "rating"]).copy()

df["user_id"] = df["user_id"].astype("int64")
df["recipe_id"] = df["recipe_id"].astype("int64")
df["rating"] = df["rating"].astype("int64")

print("Shape after essential type cleaning:", df.shape)
df.dtypes

Shape after essential type cleaning: (1132367, 5)


user_id       int64
recipe_id     int64
date         object
rating        int64
review       object
dtype: object

## 4. Parse dates for downstream chronological use
The audit already showed that dates are parseable. This step converts them into datetime format for actual modelling and later chronological splitting.

In [105]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [106]:
invalid_date_rows = df[df["date"].isna()].copy()

print("Rows with invalid parsed dates:", len(invalid_date_rows))
invalid_date_rows.head()

Rows with invalid parsed dates: 0


,user_id,recipe_id,date,rating,review


In [107]:
df = df.dropna(subset=["date"]).copy()

print("Shape after date parsing:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

Shape after date parsing: (1132367, 5)
Date range: 2000-01-25 00:00:00 to 2018-12-20 00:00:00


## 5. Confirm duplicate behaviour and apply deduplication rule
The audit found no full-row duplicates and no duplicate user–recipe pairs.
This step confirms the interaction cleaning rule on the working dataframe and ensures consistency before modelling.

In [108]:
exact_duplicate_count = int(df.duplicated().sum())
print("Exact full-row duplicates:", exact_duplicate_count)

Exact full-row duplicates: 0


### Deduplication rule
Exact duplicate rows will be removed.
Repeated interaction history across different timestamps should be preserved if present.

In [109]:
rows_before_dedup = len(df)

df = df.drop_duplicates().copy()

rows_after_dedup = len(df)

print("Rows before deduplication:", rows_before_dedup)
print("Rows after deduplication:", rows_after_dedup)
print("Rows removed:", rows_before_dedup - rows_after_dedup)

Rows before deduplication: 1132367
Rows after deduplication: 1132367
Rows removed: 0


## 6. Preserve review text without modelling it
The audit showed that review missingness is negligible.
Review text will be retained as auxiliary information, but it will not be used as a modelling feature in this phase.

In [110]:
df["review"] = df["review"].astype("string")
df["review"] = df["review"].str.strip()
df["review"] = df["review"].replace("", pd.NA)

df["review_exists"] = df["review"].notna().astype("int8")

print("Rows with review text:", int(df["review_exists"].sum()))
print("Rows without review text:", int((df["review_exists"] == 0).sum()))

Rows with review text: 1132195
Rows without review text: 172


## 7. Define the treatment of `rating = 0`
The audit showed that `rating = 0` is a non-standard value and should not automatically be interpreted as an explicit negative rating.

Preprocessing decision:
- retain `rating = 0` as an observed interaction
- exclude it from the explicit-rating target
- keep it in the implicit-feedback representation

In [111]:
rating_distribution = (
    df["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)

rating_distribution["pct"] = (rating_distribution["count"] / len(df) * 100).round(2)
rating_distribution

,rating,count,pct
0,0,60847,5.37
1,1,12818,1.13
2,2,14123,1.25
3,3,40855,3.61
4,4,187360,16.55
5,5,816364,72.09


In [112]:
zero_rating_count = int((df["rating"] == 0).sum())
zero_rating_pct = round(zero_rating_count / len(df) * 100, 2)

print("rating = 0 count:", zero_rating_count)
print("rating = 0 percentage:", zero_rating_pct)

rating = 0 count: 60847
rating = 0 percentage: 5.37


In [113]:
df["is_unrated_observation"] = (df["rating"] == 0).astype("int8")
df["explicit_rating"] = df["rating"].where(df["rating"].between(1, 5), pd.NA)
df["implicit_feedback"] = 1

## 8. Create modelling views before any filtering
The full cleaned dataset is first preserved so that filtering can be evaluated separately as a modelling trade-off rather than being hard-coded into the main cleaned output.

In [114]:
df_clean = df.sort_values(["date", "user_id", "recipe_id"]).reset_index(drop=True).copy()
df_clean.head()

,user_id,recipe_id,date,rating,review,review_exists,is_unrated_observation,explicit_rating,implicit_feedback
0,2008,992,2000-01-25,5,better than any you can get at a restaurant!,1,0,5.0,1
1,2008,3603,2000-01-25,4,better than having to actually GO to a Hooters...,1,0,4.0,1
2,2046,517,2000-02-25,5,thought this was terrific!,1,0,5.0,1
3,2046,4523,2000-02-25,2,i think i did something wrong because i could ...,1,0,2.0,1
4,2046,4684,2000-02-25,5,this is absolutely delicious. i even served i...,1,0,5.0,1


In [115]:
df_explicit_full = df_clean[df_clean["explicit_rating"].notna()].copy()
df_explicit_full["explicit_rating"] = df_explicit_full["explicit_rating"].astype("int64")

df_implicit_full = df_clean[
    [
        "user_id",
        "recipe_id",
        "date",
        "rating",
        "review",
        "review_exists",
        "is_unrated_observation",
        "implicit_feedback",
    ]
].copy()

print("Clean full dataset shape:", df_clean.shape)
print("Explicit full dataset shape:", df_explicit_full.shape)
print("Implicit full dataset shape:", df_implicit_full.shape)

Clean full dataset shape: (1132367, 9)
Explicit full dataset shape: (1071520, 9)
Implicit full dataset shape: (1132367, 8)


## 9. Evaluate low-signal filtering as a modelling option
The audit showed extreme sparsity and strong long-tail behaviour. Filtering very low-activity users and items may improve model robustness, but it can also remove a substantial amount of data.

Filtering is therefore evaluated here as an optional modelling choice rather than a fixed cleaning rule.

In [116]:
user_activity = df_clean.groupby("user_id").size().rename("interaction_count")
item_popularity = df_clean.groupby("recipe_id").size().rename("interaction_count")

print("Median interactions per user:", user_activity.median())
print("Median interactions per item:", item_popularity.median())
print("Users with 1 interaction:", int((user_activity == 1).sum()))
print("Items with 1 interaction:", int((item_popularity == 1).sum()))

Median interactions per user: 1.0
Median interactions per item: 2.0
Users with 1 interaction: 166256
Items with 1 interaction: 91953


In [117]:
def iterative_filter(data: pd.DataFrame, min_user_interactions: int, min_item_interactions: int) -> pd.DataFrame:
    filtered = data.copy()
    previous_shape = None

    while previous_shape != filtered.shape:
        previous_shape = filtered.shape

        valid_users = (
            filtered["user_id"]
            .value_counts()
            .loc[lambda s: s >= min_user_interactions]
            .index
        )
        filtered = filtered[filtered["user_id"].isin(valid_users)]

        valid_items = (
            filtered["recipe_id"]
            .value_counts()
            .loc[lambda s: s >= min_item_interactions]
            .index
        )
        filtered = filtered[filtered["recipe_id"].isin(valid_items)]

    return filtered

In [118]:
threshold_grid = [
    (1, 1),
    (2, 2),
    (3, 3),
    (5, 5),
]

filtering_results = []

for min_user_interactions, min_item_interactions in threshold_grid:
    filtered_tmp = iterative_filter(
        df_clean,
        min_user_interactions=min_user_interactions,
        min_item_interactions=min_item_interactions,
    )

    filtering_results.append(
        {
            "min_user_interactions": min_user_interactions,
            "min_item_interactions": min_item_interactions,
            "rows_remaining": int(len(filtered_tmp)),
            "rows_retained_pct": round(len(filtered_tmp) / len(df_clean) * 100, 2),
            "users_remaining": int(filtered_tmp["user_id"].nunique()),
            "items_remaining": int(filtered_tmp["recipe_id"].nunique()),
        }
    )

filtering_results_df = pd.DataFrame(filtering_results)
filtering_results_df

,min_user_interactions,min_item_interactions,rows_remaining,rows_retained_pct,users_remaining,items_remaining
0,1,1,1132367,100.00,226570,231637
1,2,2,870419,76.87,56492,127119
2,3,3,733951,64.82,32635,80511
3,5,5,555618,49.07,17813,41240


### Filtering interpretation
This comparison is used to decide whether filtering should be applied now, delayed until model-specific experiments, or kept very light.
If stricter thresholds remove too much of the already sparse interaction data, the full cleaned dataset may be more appropriate as the main Phase 2 output.

In [119]:
df_filtered_optional = iterative_filter(df_clean, 2, 2).copy()

print("Optional filtered shape (2,2):", df_filtered_optional.shape)
print("Optional filtered retention %:", round(len(df_filtered_optional) / len(df_clean) * 100, 2))

Optional filtered shape (2,2): (870419, 9)
Optional filtered retention %: 76.87


## 11. Build a compact cleaning summary
A compact summary is saved for reporting and later write-up use.

In [120]:
cleaning_summary = pd.DataFrame([
    {
        "raw_rows": int(len(df_raw)),
        "rows_after_core_cleaning": int(len(df_clean)),
        "explicit_rows_full": int(len(df_explicit_full)),
        "implicit_rows_full": int(len(df_implicit_full)),
        "rating_zero_count": int(df_clean["is_unrated_observation"].sum()),
        "rows_with_review_text": int(df_clean["review_exists"].sum()),
        "min_date": str(df_clean["date"].min()),
        "max_date": str(df_clean["date"].max()),
        "optional_filtered_rows_2_2": int(len(df_filtered_optional)),
        "optional_filtered_retention_pct_2_2": round(len(df_filtered_optional) / len(df_clean) * 100, 2),
    }
])

cleaning_summary

,raw_rows,rows_after_core_cleaning,explicit_rows_full,implicit_rows_full,rating_zero_count,rows_with_review_text,min_date,max_date,optional_filtered_rows_2_2,optional_filtered_retention_pct_2_2
0,1132367,1132367,1071520,1132367,60847,1132195,2000-01-25 00:00:00,2018-12-20 00:00:00,870419,76.87


In [121]:
rating_distribution.to_csv(TABLES_DIR / "02_rating_distribution.csv", index=False)
filtering_results_df.to_csv(TABLES_DIR / "02_filtering_threshold_comparison.csv", index=False)
cleaning_summary.to_csv(TABLES_DIR / "02_interaction_cleaning_summary.csv", index=False)

## 12. Save cleaned outputs
The core cleaned datasets are saved as outputs.

An optional light-filtered variant is also saved for later model experiments if needed.

In [122]:
df_clean.to_parquet(PROCESSED_DIR / "interactions_clean.parquet", index=False)
df_explicit_full.to_parquet(PROCESSED_DIR / "interactions_explicit.parquet", index=False)
df_implicit_full.to_parquet(PROCESSED_DIR / "interactions_implicit.parquet", index=False)
df_filtered_optional.to_parquet(PROCESSED_DIR / "interactions_filtered_optional_2_2.parquet", index=False)

print("Saved cleaned interaction datasets.")

Saved cleaned interaction datasets.


## 13. Final decisions

The following core interaction-cleaning decisions were applied:

- essential numeric fields were standardised
- dates were converted to datetime format
- exact duplicate interaction rows were removed if present
- review text was preserved but not modelled
- `rating = 0` was treated as an observed but unrated interaction
- separate explicit and implicit interaction datasets were created

Low-signal filtering was evaluated as a modelling-oriented preprocessing option rather than fixed as a mandatory cleaning rule. Threshold testing showed that stricter iterative filtering can remove a substantial proportion of the already sparse interaction data, so the full cleaned dataset is retained as output, while lighter filtered variants can be used later if required.

The next step is to move the validated core cleaning logic into `src/data/clean_interactions.py`.